In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

In [2]:
import numpy as np
import cv2 
from tensorflow import keras

model = keras.models.load_model("C:\\Users\\huynh\\Documents\\ML\\models\\keobuabao.keras")
class_name = ["Bao", "Bua", "Keo"]

img_path = "C:\\Users\\huynh\\Pictures\\IMG_20260219_181941.jpg"
img = cv2.imread(img_path)

img_resize = cv2.resize(img, (224,224))
img_color = cv2.cvtColor(img_resize, cv2.COLOR_RGB2BGR)

img_input = np.expand_dims(img_color, axis = 0)

predict = model.predict(img_input, verbose = 0)
label = class_name[np.argmax(predict)]
score = np.max(predict)

print(f"ket qua: {label}, ti le: {score}")

ket qua: Bao, ti le: 0.42574989795684814


In [8]:
import cv2
import numpy as np
from tensorflow import keras

# 1. Tải mô hình (Chỉ chạy 1 lần duy nhất ở ngoài cùng)
model = keras.models.load_model('models/keobuabao1.keras')
class_names = ['Bao', 'Bua', 'Keo'] 

# 2. Khởi tạo Camera
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Không thể kết nối với camera.")
        break

    # --- BƯỚC 1: TIỀN XỬ LÝ ẢNH CHUẨN XÁC ---
    h, w, _ = frame.shape
    max_dim = max(h, w)
    
    # Tính toán lượng viền đen cần bù để tạo ảnh vuông (Pad to aspect ratio)
    top = (max_dim - h) // 2
    bottom = max_dim - h - top
    left = (max_dim - w) // 2
    right = max_dim - w - left
    
    # Chèn viền đen vào khung hình gốc
    square_frame = cv2.copyMakeBorder(frame, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    
    # Thu nhỏ về đúng kích thước đầu vào của model
    img_resized = cv2.resize(square_frame, (224, 224))
    
    # Đổi hệ màu từ BGR (OpenCV) sang RGB (TensorFlow)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    # Thêm chiều Batch (tạo thành ma trận 1, 224, 224, 3)
    img_input = np.expand_dims(img_rgb, axis=0)

    # --- BƯỚC 2: DỰ ĐOÁN ---
    # Đặt verbose=0 để tránh việc in log liên tục làm chậm vòng lặp
    prediction = model.predict(img_input, verbose=0)
    score = np.max(prediction)
    label = class_names[np.argmax(prediction)]

    # --- BƯỚC 3: HIỂN THỊ KẾT QUẢ ---
    # Chỉnh màu chữ: Xanh lá nếu tự tin cao, Đỏ nếu tự tin thấp
    color = (0, 255, 0) if score > 0.7 else (0, 0, 255)
    
    # Vẽ chữ lên khung hình GỐC (chưa chèn viền) để nhìn tự nhiên hơn
    cv2.putText(frame, f"{label}: {score*100:.2f}%", (30, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    
    # Hiện cửa sổ camera
    cv2.imshow('Nhan Dien Keo Bua Bao', frame)
    
    # Bấm phím 'q' trên bàn phím để thoát
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Giải phóng tài nguyên phần cứng
cap.release()
cv2.destroyAllWindows()